# 

In [3]:
import os
import pandas as pd
import torch

# 1. Checar GPU
print("GPU ativa:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Placa:", torch.cuda.get_device_name(0))

# 2. Caminho exato dos dados
base_path = "/kaggle/input/competitions/llm-classification-finetuning"

# 3. Carregar datasets
train_df = pd.read_csv(f"{base_path}/train.csv")
test_df = pd.read_csv(f"{base_path}/test.csv")

print(f"\nTreino carregado com sucesso! Linhas e colunas: {train_df.shape}")
print(f"Teste carregado com sucesso! Linhas e colunas: {test_df.shape}")
print("\nPrimeiras linhas do treino:")
train_df[['prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']].head(2)

GPU ativa: True
Placa: Tesla T4

Treino carregado com sucesso! Linhas e colunas: (57477, 9)
Teste carregado com sucesso! Linhas e colunas: (3, 4)

Primeiras linhas do treino:


,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


In [4]:
import numpy as np
import torch
from transformers import AutoTokenizer
from datasets import Dataset

# 1. Definir o modelo base (usaremos a versão small para treino rápido e eficiente na T4)
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 512  # Limite de tokens para otimizar velocidade e memória VRAM

print(f"Carregando tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Tratar valores nulos
train_df['prompt'] = train_df['prompt'].fillna("")
train_df['response_a'] = train_df['response_a'].fillna("")
train_df['response_b'] = train_df['response_b'].fillna("")

# 3. Mapear as classes binárias para inteiros (0: Winner A, 1: Winner B, 2: Tie)
def get_label(row):
    if row['winner_model_a'] == 1:
        return 0
    elif row['winner_model_b'] == 1:
        return 1
    else:
        return 2

train_df['label'] = train_df.apply(get_label, axis=1)

# 4. Formatar o texto de entrada como Cross-Encoder
# Formato: "Prompt: <prompt> [SEP] Response A: <resp_a> [SEP] Response B: <resp_b>"
train_df['input_text'] = (
    "Prompt: " + train_df['prompt'] + 
    " " + tokenizer.sep_token + " Response A: " + train_df['response_a'] + 
    " " + tokenizer.sep_token + " Response B: " + train_df['response_b']
)

# 5. Função de Tokenização com truncamento e padding dinâmico
def tokenize_function(examples):
    return tokenizer(
        examples['input_text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False # O padding dinâmico será feito pelo DataCollator durante o batch
    )

# 6. Converter o DataFrame do Pandas para Hugging Face Dataset
print("\nConvertendo para Hugging Face Dataset e tokenizando...")
hf_dataset = Dataset.from_pandas(train_df[['input_text', 'label']])
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, remove_columns=['input_text'])

# 7. Separar em treino (85%) e validação (15%) com semente fixa
split_dataset = tokenized_dataset.train_test_split(test_size=0.15, seed=42)

print("\n--- Estrutura Final dos Dados ---")
print("Exemplos no Treino:", len(split_dataset['train']))
print("Exemplos na Validação:", len(split_dataset['test']))
print("Colunas prontas para o PyTorch:", split_dataset['train'].column_names)

Carregando tokenizer: microsoft/deberta-v3-small...


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


Convertendo para Hugging Face Dataset e tokenizando...


Map:   0%|          | 0/57477 [00:00<?, ? examples/s]


--- Estrutura Final dos Dados ---
Exemplos no Treino: 48855
Exemplos na Validação: 8622
Colunas prontas para o PyTorch: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


# 

In [5]:
import torch
import numpy as np
from scipy.special import softmax
from sklearn.metrics import log_loss
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# 1. Carregar Modelo
MODEL_NAME = "microsoft/deberta-v3-small"
num_labels = 3
print(f"Carregando modelo {MODEL_NAME}...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={0: "winner_model_a", 1: "winner_model_b", 2: "winner_tie"},
    label2id={"winner_model_a": 0, "winner_model_b": 1, "winner_tie": 2}
)

# 2. Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3. Métrica Oficial (Log Loss com float64 e classes inteiras)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    logits = np.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0).astype(np.float64)
    probabilities = softmax(logits, axis=-1)
    probabilities = np.clip(probabilities, 1e-15, 1.0 - 1e-15)
    probabilities = probabilities / probabilities.sum(axis=1, keepdims=True)
    
    loss = log_loss(labels.astype(np.int64), probabilities, labels=[0, 1, 2])
    return {"log_loss": loss}

# 4. Hiperparâmetros de Treinamento
training_args = TrainingArguments(
    output_dir="./results_deberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=300,
    max_grad_norm=1.0,
    adam_epsilon=1e-6,
    fp16=False,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="log_loss",
    greater_is_better=False,
    report_to="none"
)

# 5. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 6. Rodar Treinamento
print("\nIniciando o Fine-Tuning do DeBERTa-v3...")
trainer.train()

Carregando modelo microsoft/deberta-v3-small...


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3054, training_loss=4.409261682256813, metrics={'train_runtime': 1886.0022, 'train_samples_per_second': 51.808, 'train_steps_per_second': 1.619, 'total_flos': 1.294408140327936e+16, 'train_loss': 4.409261682256813, 'epoch': 2.0})

In [7]:
import pandas as pd
from scipy.special import softmax
from datasets import Dataset

print("Preparando conjunto de teste para inferência...")

# 1. Tratar nulos e formatar no padrão Cross-Encoder
test_df['prompt'] = test_df['prompt'].fillna("")
test_df['response_a'] = test_df['response_a'].fillna("")
test_df['response_b'] = test_df['response_b'].fillna("")

test_df['input_text'] = (
    "Prompt: " + test_df['prompt'] + 
    " " + tokenizer.sep_token + " Response A: " + test_df['response_a'] + 
    " " + tokenizer.sep_token + " Response B: " + test_df['response_b']
)

# 2. Tokenizar os dados de teste
hf_test_dataset = Dataset.from_pandas(test_df[['input_text']])
tokenized_test = hf_test_dataset.map(tokenize_function, batched=True, remove_columns=['input_text'])

# 3. Predição com o melhor checkpoint salvo (Época 2)
print("Gerando probabilidades com o modelo finetunado...")
predictions = trainer.predict(tokenized_test)
test_probabilities = softmax(predictions.predictions, axis=-1)

# 4. Formatar submissão
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': test_probabilities[:, 0],
    'winner_model_b': test_probabilities[:, 1],
    'winner_tie': test_probabilities[:, 2]
})

# 5. Salvar CSV
submission.to_csv('submission_deberta.csv', index=False)
print("\nArquivo 'submission_deberta.csv' gerado com sucesso!\n")
print(submission.head())

Preparando conjunto de teste para inferência...


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Gerando probabilidades com o modelo finetunado...



Arquivo 'submission_deberta.csv' gerado com sucesso!

        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.341553        0.344238    0.313965
1   211333        0.341553        0.344238    0.313965
2  1233961        0.341553        0.344238    0.313965


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [8]:
import pandas as pd

# Ler o CSV gerado
df_check = pd.read_csv('submission_deberta.csv')

print(f"Dimensões do arquivo: {df_check.shape}")
print("\nPrimeiras linhas geradas:")
print(df_check.to_string())

# Verificar se as probabilidades somam ~1.0
print("\nSoma das probabilidades por linha:")
print(df_check[['winner_model_a', 'winner_model_b', 'winner_tie']].sum(axis=1))

Dimensões do arquivo: (3, 4)

Primeiras linhas geradas:
        id  winner_model_a  winner_model_b  winner_tie
0   136060          0.3416          0.3442       0.314
1   211333          0.3416          0.3442       0.314
2  1233961          0.3416          0.3442       0.314

Soma das probabilidades por linha:
0    0.9998
1    0.9998
2    0.9998
dtype: float64
